In [ ]:
# -------------------------------
# 1️⃣ Import Required Libraries
# -------------------------------
import pandas as pd                     # For data handling
import numpy as np                      # For numerical operations
import matplotlib.pyplot as plt         # For plotting
import seaborn as sns                   # For advanced visualization

from sklearn.model_selection import train_test_split, cross_val_score  # Train/test split and cross-validation
from sklearn.preprocessing import StandardScaler                          # Feature scaling
from sklearn.linear_model import LogisticRegression                       # Logistic Regression
from sklearn.tree import DecisionTreeClassifier                            # Decision Tree
from sklearn.ensemble import RandomForestClassifier                        # Random Forest
from sklearn.svm import SVC                                                # Support Vector Machine
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix  # Metrics

import joblib   # Save/load trained models

# -------------------------------
# 2️⃣ Load the Dataset
# -------------------------------
data = pd.read_csv("merged_heart_dataset.csv")  # Load your dataset
print(data.head())                               # Display first 5 rows to inspect

# -------------------------------
# 3️⃣ Handle Missing Values
# -------------------------------
# Fill missing values with median (robust to outliers)
data.fillna(data.median(), inplace=True)

# Verify all missing values handled
print(data.isnull().sum())

# -------------------------------
# 4️⃣ Prepare Features and Target
# -------------------------------
# Convert 'num' column to binary target: 0 = no disease, 1 = disease
data['num'] = data['num'].apply(lambda x: 1 if x > 0 else 0)

# Drop 'id' and 'dataset' columns from features
X = data.drop(columns=['id', 'dataset', 'num'])
y = data['num']

print("Feature columns:", X.columns)
print("Target distribution:\n", y.value_counts())

# -------------------------------
# 5️⃣ Split Dataset into Training and Testing
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# stratify=y ensures both train/test have same ratio of disease/no disease

# -------------------------------
# 6️⃣ Feature Scaling
# -------------------------------
scaler = StandardScaler()           # Initialize scaler
X_train = scaler.fit_transform(X_train)   # Fit and scale training data
X_test = scaler.transform(X_test)         # Scale test data using same parameters

# -------------------------------
# 7️⃣ Initialize Models
# -------------------------------
log_model = LogisticRegression()
tree_model = DecisionTreeClassifier()
rf_model = RandomForestClassifier()
svm_model = SVC(probability=True)

# -------------------------------
# 8️⃣ Train the Models
# -------------------------------
log_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)
svm_model.fit(X_train, y_train)

# -------------------------------
# 9️⃣ Make Predictions
# -------------------------------
log_pred = log_model.predict(X_test)
tree_pred = tree_model.predict(X_test)
rf_pred = rf_model.predict(X_test)
svm_pred = svm_model.predict(X_test)

# -------------------------------
# 🔟 Evaluate Model Accuracy
# -------------------------------
print("Logistic Regression Accuracy:", accuracy_score(y_test, log_pred))
print("Decision Tree Accuracy:", accuracy_score(y_test, tree_pred))
print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print("SVM Accuracy:", accuracy_score(y_test, svm_pred))

# -------------------------------
# 1️⃣1️⃣ Confusion Matrix for Random Forest
# -------------------------------
cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Random Forest")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# -------------------------------
# 1️⃣2️⃣ Detailed Classification Report
# -------------------------------
print("Classification Report - Random Forest:\n")
print(classification_report(y_test, rf_pred))

# -------------------------------
# 1️⃣3️⃣ Cross-Validation Scores
# -------------------------------
models = [log_model, svm_model, tree_model, rf_model]
model_names = ["Logistic Regression", "SVM", "Decision Tree", "Random Forest"]

for model, name in zip(models, model_names):
    scores = cross_val_score(model, X, y, cv=6)  # 6-fold cross-validation
    print(f"{name} Cross-Validation Scores: {scores}")
    print(f"{name} Average Score: {scores.mean():.4f}\n")

# -------------------------------
# 1️⃣4️⃣ Save the Best Model
# -------------------------------
joblib.dump(rf_model, "heart_disease_model.pkl")
print("Random Forest model saved as 'heart_disease_model.pkl'")